In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pyraws
from pyraws.raw.raw_event import Raw_event
import warnings
import torch
import cv2
import os
import zipfile
import shutil

In [ ]:
# THRawS dataset
# Using pyRaws to read the .bin files
# The .bin files might be in different sizes etc. and they form 
# long series of binaries. We need pyRaws to read and convert the files
# https://www.youtube.com/watch?v=aLe7Pw3DVmo (video from Roberto)

warnings.filterwarnings("ignore", category=UserWarning, module="pyraws")

l0_folder = '/home/laura/Scriptie/data/temp_thraws/L0_Data_Uitgepakt/Chillan_Nevados_de_00_NE'

try:
    raw_data = pyraws.Raw_event()
    target_bands = ['B04', 'B08', 'B11']
    raw_data.from_path(l0_folder, target_bands)
    print(f"Successfully read L0 folder with pyRaws: {l0_folder}")
    granule_names = raw_data.get_granules_names()
    print(f"Granules found: {granule_names}")
except Exception as e:
    print(f"An error occurred while reading the L0 folder with pyRaws: {e}")

# Check shape of the B08 band to confirm we have the data in matrix form
if len(granule_names) > 0:
    tile = raw_data.get_granule(0)
    band_b08 = tile.get_band('B08')
    if hasattr(band_b08, 'shape'):
        print(f"Matrix B08 Shape: {band_b08.shape}")
    elif hasattr(band_b08, 'pixels'):
        print(f"Matrix B08 Shape: {band_b08.pixels.shape}")

In [ ]:
def get_numpy_image(band_name):
    band_data = tile.get_band(band_name)
    
    if hasattr(band_data, 'pixels'):
        matrix = band_data.pixels
    else:
        matrix = band_data
        
    if torch.is_tensor(matrix):
        matrix = matrix.cpu().numpy()
        
    return matrix

b04_img = get_numpy_image('B04')
b08_img = get_numpy_image('B08')
b11_img = get_numpy_image('B11')

print(f"Shapes - B04: {b04_img.shape}, B08: {b08_img.shape}, B11: {b11_img.shape}")

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

def plot_band(ax, img, title):
    vmax = np.percentile(img, 98) 
    im = ax.imshow(img, cmap='gray', vmin=0, vmax=vmax)
    ax.set_title(title, fontsize=14)
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plot_band(axes[0], b04_img, 'B04 (Red - landscape)')
plot_band(axes[1], b08_img, 'B08 (NIR - Vegetation)')
plot_band(axes[2], b11_img, 'B11 (SWIR - Heat/Fire)')

plt.tight_layout()
plt.show()

Next, we will use upscaling to fit the B11 band on top of the other bands

In [ ]:
b11_resized = cv2.resize(b11_img, (b04_img.shape[1], b04_img.shape[0]), interpolation=cv2.INTER_NEAREST)

# Stack of spectral bands
X_tile = np.stack([b04_img, b08_img, b11_resized], axis=-1)


Patching

In [ ]:
raw_data_dir = '/home/laura/Scriptie/ruwe_data'
bewerkte_data_dir = '/home/laura/Scriptie/bewerkte_data/THRawS'
os.makedirs(bewerkte_data_dir, exist_ok=True)

PATCH_SIZE = 256
STRIDE = 256

patches_X = []
patches_y = []

height, width, channels = X_tile.shape

for y in range(0, height - PATCH_SIZE + 1, STRIDE):
    for x in range(0, width - PATCH_SIZE + 1, STRIDE):
        
        patch = X_tile[y:y+PATCH_SIZE, x:x+PATCH_SIZE, :]
        
        patches_X.append(patch)
        
        patches_y.append(0)

X_dataset = np.array(patches_X)
y_dataset = np.array(patches_y)

np.save(os.path.join(bewerkte_data_dir, 'X_patches_NE.npy'), X_dataset)
np.save(os.path.join(bewerkte_data_dir, 'y_patches_NE.npy'), y_dataset)

print(f"X_dataset shape: {X_dataset.shape}")
print(f"y_dataset shape: {y_dataset.shape}")

In [ ]:
path_THRawS = "/home/laura/Scriptie/data/THRawS.zip"
temp_extract_dir = '/home/laura/Scriptie/data/temp_thraws/L0_Data_Uitgepakt'
event_folder = "Australia_0/"
full_event_path = os.path.join(temp_extract_dir, "Australia_0")

if os.path.exists(full_event_path):
    shutil.rmtree(full_event_path)

try:
    with zipfile.ZipFile(path_THRawS, "r") as outer_zip:
        with outer_zip.open('1.zip') as inner_zip_file:
            with zipfile.ZipFile(inner_zip_file) as inner_zip:
                safe_files = [f for f in inner_zip.namelist() if f.startswith(event_folder)]
                inner_zip.extractall(path=temp_extract_dir, members=safe_files)
except Exception as e:
    print(f"Error: {e}")

raw_data_fire = pyraws.Raw_event()

raw_data_fire.from_path(full_event_path, ['B04', 'B08', 'B11'])
tile_fire = raw_data_fire.get_granule(0) 

def get_numpy_image(band_name, tile):
    band_data = tile.get_band(band_name)
    matrix = band_data.pixels if hasattr(band_data, 'pixels') else band_data
    if hasattr(matrix, 'cpu'): matrix = matrix.cpu().numpy()
    return matrix

b04_f = get_numpy_image('B04', tile_fire)
b08_f = get_numpy_image('B08', tile_fire)
b11_f = get_numpy_image('B11', tile_fire)

b11_resized_f = cv2.resize(b11_f, (b04_f.shape[1], b04_f.shape[0]), interpolation=cv2.INTER_NEAREST)
X_tile_fire = np.stack([b04_f, b08_f, b11_resized_f], axis=-1)

PATCH_SIZE = 256
STRIDE = 256
patches_X_fire = []

for y in range(0, X_tile_fire.shape[0] - PATCH_SIZE + 1, STRIDE):
    for x in range(0, X_tile_fire.shape[1] - PATCH_SIZE + 1, STRIDE):
        patches_X_fire.append(X_tile_fire[y:y+PATCH_SIZE, x:x+PATCH_SIZE, :])

X_fire_dataset = np.array(patches_X_fire)

max_heat_per_patch = [np.max(patch[:, :, 2]) for patch in X_fire_dataset]
treshold = np.percentile(max_heat_per_patch, 95) 
y_fire_dataset = np.array([1 if hitte >= treshold else 0 for hitte in max_heat_per_patch])


X_incomplete = np.load(os.path.join(bewerkte_data_dir, 'X_patches_NE.npy'))
y_incomplete = np.load(os.path.join(bewerkte_data_dir, 'y_patches_NE.npy'))

X_complete = np.concatenate((X_incomplete, X_fire_dataset), axis=0)
y_complete = np.concatenate((y_incomplete, y_fire_dataset), axis=0)

np.save(os.path.join(bewerkte_data_dir, 'X_patches_complete.npy'), X_complete)
np.save(os.path.join(bewerkte_data_dir, 'y_patches_complete.npy'), y_complete)